In [157]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
# from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
import operator

load_dotenv()

True

In [158]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for eassy")
    score: int = Field(description="Score for eassy out of 10", ge=0, le=10)

In [159]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     temperature=0,
# )

llm = ChatOllama(
    model="llama3.1",
    temperature=0,
)


structured_llm = llm.with_structured_output(EvaluationSchema)

### Workflow

In [ ]:
class EassyState(TypedDict):
    eassy: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]       # Reducer
    avg_score: float

In [161]:
def evaluate_language(state: EassyState):
    prompt = f"Evaluate the language quality of the following eassy and provide a feedback and assign a score out of 10. /n{eassy}"
    output = structured_llm.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}


def evaluate_analysis(state: EassyState):
    prompt = f"Evaluate the analysis quality of the following eassy and provide a feedback and assign a score out of 10. /n{eassy}"
    output = structured_llm.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}


def evaluate_clarity(state: EassyState):
    prompt = f"Evaluate the clarity of thoughts of the following eassy and provide a feedback and assign a score out of 10. /n{eassy}"
    output = structured_llm.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}


def final_evaluation(state: EassyState):
    prompt = f"""Based on the following feedbacks, create a summarized feedback.
    language feedback: {state['language_feedback']}
    analysis_feedback: {state['analysis_feedback']}
    clarity_feedback: {state['clarity_feedback']}"""

    overall_feedback = llm.invoke(prompt).content


    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [162]:
graph = StateGraph(EassyState)


graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)


graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')
graph.add_edge('final_evaluation', END)


workflow = graph.compile()

In [163]:
eassy = """Future of AI in Pakistan

Artificial Intelligence represents both a critical strategic necessity and a powerful engine for progress in Pakistan, offering the potential to leapfrog traditional developmental stages despite socio-economic challenges. Driven by a massive youth demographic and a growing technology sector, the national landscape is uniquely positioned to integrate data-driven innovations into core industries.

In agriculture, which remains the backbone of the economy, AI tools are transforming traditional farming through precision analytics and satellite imaging that optimize water usage, monitor crop health, and predict weather disruptions. In the healthcare sector, machine learning algorithms and diagnostic vision models help bridge the severe gap in medical coverage by aiding doctors in low-resource rural clinics with early disease detection. Furthermore, Pakistan's software and freelance workforce is shifting from basic task outsourcing toward high-value AI development, natural language processing, and automated service solutions, significantly boosting digital exports and energizing local technology hubs across major urban centers.

However, realizing this potential requires addressing significant structural hurdles. The nation continues to face an unreliable energy grid, limited local data center infrastructure, and a pronounced digital divide between urban and rural populations. Furthermore, a steady brain drain of top technical talent seeking better opportunities abroad limits local research capacity, while regulatory frameworks for data privacy and ethical AI implementation remain in early stages.

To build a sustainable technological ecosystem, Pakistan must focus on upgrading university curricula, establishing public-private high-performance computing centers, and implementing clear data governance policies. If approached with deliberate policy, targeted education, and localized problem-solving, artificial intelligence can serve as a cornerstone for economic growth, improved public services, and long-term technological sovereignty across the country."""

In [164]:
initial_state = {'eassy': eassy}

final_state = workflow.invoke(initial_state)

In [165]:
print(final_state['individual_scores'])
print(final_state['avg_score'])

print(final_state['language_feedback'])
print(final_state['analysis_feedback'])
print(final_state['clarity_feedback'])

[8, 8, 8]
8.0
The essay provides a comprehensive overview of the potential of Artificial Intelligence (AI) in Pakistan, highlighting both its benefits and challenges. The language is clear and concise, making it easy to understand for a general audience. However, there are some areas that require improvement to elevate the quality of the essay.
The essay provides an insightful analysis of the potential of Artificial Intelligence (AI) in Pakistan's development. It highlights both the opportunities and challenges that come with integrating AI into various sectors such as agriculture, healthcare, and technology. The writer effectively uses specific examples to illustrate how AI can transform traditional industries and improve public services. However, there are areas for improvement in terms of clarity, coherence, and depth of analysis.
The essay provides a clear and concise overview of the potential impact of Artificial Intelligence (AI) on Pakistan's economy and society. The author effe